# R1-06 Timing + cost + timeline
Outputs: `data/timing_log.csv`, `cost_estimation/cost_analysis.md`, `cost_estimation/timeline_estimate.md`.
Cells: 1 Check, 2 Backup, 3 Create, 4 Verify, 5 Unchanged, 6 Undo (RUN_UNDO = False), 7 Git (print only).
All numbers are computed, cited, or labelled estimate / synthetic.

In [1]:
# 1 Check: right folder, input files exist, 10 timing emails are all held-out
from pathlib import Path
import pandas as pd

REPO = Path.cwd() if (Path.cwd() / 'AGENTS.md').exists() else Path.cwd().parent

def ok(name, cond):
    print('PASS' if cond else 'FAIL', '-', name)
    return cond

# inputs only (timing_log.csv is an output, it is backed up in cell 2)
INPUTS = ['langsmith/trace_summary.csv', 'data/processed/spreekorb_emails_60.csv',
          'langsmith/monitoring_note.md', 'langsmith/human_review.csv']
ok('repo folder has AGENTS.md', (REPO / 'AGENTS.md').exists())
for p in INPUTS:
    ok('input exists: ' + p, (REPO / p).exists())

em = pd.read_csv(REPO / 'data/processed/spreekorb_emails_60.csv', dtype=str).fillna('')
tim = em[em['timing_flag'].astype(str).str.lower() == 'true']
TIMING_IDS = sorted(tim['email_id'])
ok('10 timing emails', len(TIMING_IDS) == 10)
ok('all timing emails are heldout', (tim['split'] == 'heldout').all())
print(TIMING_IDS)

# optional input: Shopzilla raw file (read-only), used only to cross-check the manual time
RAW = REPO / 'data/raw/shopzilla_cs_satisfaction.csv'
RAW_OK = RAW.exists()
if RAW_OK:
    print('PASS - Shopzilla raw file exists')
else:
    print('WARN - Shopzilla raw file missing, cross-check skipped')

PASS - repo folder has AGENTS.md
PASS - input exists: langsmith/trace_summary.csv
PASS - input exists: data/processed/spreekorb_emails_60.csv
PASS - input exists: langsmith/monitoring_note.md
PASS - input exists: langsmith/human_review.csv
PASS - 10 timing emails
PASS - all timing emails are heldout
['S05', 'S11', 'S20', 'S24', 'S40', 'S44', 'S47', 'S52', 'S57', 'S60']
PASS - Shopzilla raw file exists


In [2]:
# 2 Backup: copy every file we will change to _backup/<timestamp>/, hash files we do not own
import shutil, hashlib
from datetime import datetime

OUTPUTS = ['data/timing_log.csv', 'cost_estimation/cost_analysis.md', 'cost_estimation/timeline_estimate.md']
PROTECTED = ['langsmith/trace_summary.csv', 'data/processed/spreekorb_emails_60.csv',
             'langsmith/monitoring_note.md', 'langsmith/human_review.csv']
if RAW_OK:
    PROTECTED.append('data/raw/shopzilla_cs_satisfaction.csv')   # read-only input

TS = datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
EXISTED = {}
for p in OUTPUTS:
    EXISTED[p] = (REPO / p).exists()
    if EXISTED[p]:
        (BK / p).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(REPO / p, BK / p)

def sha(p):
    return hashlib.sha256((REPO / p).read_bytes()).hexdigest()

HASH_BEFORE = {p: sha(p) for p in PROTECTED}
print('Backup folder:', BK)
print('Existed before:', EXISTED)

Backup folder: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261127_144753
Existed before: {'data/timing_log.csv': True, 'cost_estimation/cost_analysis.md': False, 'cost_estimation/timeline_estimate.md': False}


In [3]:
# 3 Create: timing log + cost analysis + timeline.
# Inputs live in the A dict (value, unit, label, source). Results are computed here, printed and written to the md files.
NL = chr(10)
KAGGLE = 'https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction'

A = {
 'usd_eur': (0.9123, 'EUR per USD', 'estimate', 'tradingview.com USDEUR, 27 Nov 2026 (as in langsmith/monitoring_note.md)'),
 'rate_h': (95, 'EUR/h', 'estimate', 'IT freelancer median, freelancermap Freelancer-Kompass 2026, https://www.freelancermap.de/blog/stundensatz-it-freelancer/'),
 'hours_day': (8, 'h/day', 'estimate', 'standard working day, own assumption (no source)'),
 'days_prototype': (10, 'days', 'estimate', 'own estimate, no source (Round 1 prototype, done, sunk cost)'),
 'days_pilot': (15, 'days', 'estimate', 'own estimate, no source (n8n + mailbox + review UI + DPA / works-council prep)'),
 'days_rollout': (10, 'days', 'estimate', 'own estimate, no source'),
 'weeks_prototype': (2, 'weeks', 'estimate', 'own estimate, no source (done in Round 1)'),
 'weeks_pilot': (6, 'weeks', 'estimate', 'own estimate, no source'),
 'weeks_rollout': (8, 'weeks', 'estimate', 'own estimate, no source'),
 'volume': (3017, 'emails/month', 'computed (Shopzilla proxy, busiest month)', 'research/sector_research.md; Shopzilla data ' + KAGGLE + ' (real-data proxy, not Spreekorb)'),
 'timing_set': (len(TIMING_IDS), 'emails', 'synthetic', 'data/processed/spreekorb_emails_60.csv, rows with timing_flag, split heldout (synthetic set)'),
 'manual_min': (15, 'min/email', 'estimate', 'estimate (Ayhan), revised from 20 to 15 min on 27 Nov 2026 after the Shopzilla cross-check; no measured basis'),
 'ai_min': (5, 'min/email', 'measured (self-reported average, Ayhan)', 'Ayhan, 27 Nov 2026; average, done second on the same emails (learning bias)'),
 'n8n_eur': (60, 'EUR/month', 'estimate', 'https://n8n.io/pricing (Pro about 50 EUR billed annually, 60 monthly; third-party sites differ, check before buying)'),
 'n8n_limit': (10000, 'executions/month', 'estimate', 'https://n8n.io/pricing (Pro plan)'),
 'n8n_exec_per_email': (1, 'executions/email', 'estimate', 'own assumption: one email = one n8n execution'),
 'ls_usd': (39, 'USD/seat/month', 'estimate', 'LangSmith Plus, https://www.langchain.com/pricing (10k base traces included)'),
 'ls_seats': (2, 'seats', 'estimate', 'own estimate, no source'),
 'at_usd': (20, 'USD/seat/month', 'estimate', 'Airtable Team, annual billing, https://support.airtable.com/docs/airtable-plans'),
 'at_seats': (3, 'editors', 'estimate', 'own estimate, no source'),
 'tableau_eur': (0, 'EUR/month', 'estimate', 'Tableau Public, public data only, https://public.tableau.com/app/about'),
 'salary': (2650, 'EUR/month gross', 'estimate', 'mid of 2,400-2,900; https://www.profiling-institut.de/ausbildungsberufe/kaufmann-dialogmarketing/ (BA Entgeltatlas median 2,922), https://www.gehalt.de/beruf/kaufmann-frau-fuer-dialogmarketing (about 2,460)'),
 'overhead': (1.2, 'factor', 'estimate', 'employer cost on top of gross, own estimate, no source'),
 'hours_month': (160, 'h/month', 'estimate', 'paid hours per month, own estimate, no source'),
 'queue_match': (85, '% (n=20)', 'synthetic', 'langsmith/monitoring_note.md (held-out, synthetic emails)'),
}
v = {k: A[k][0] for k in A}
man, ai, vol = v['manual_min'], v['ai_min'], v['volume']

# re-routing share = 100 - queue_match
re_route = 100 - v['queue_match']
A['re_route'] = (re_route, '% (n=20)', 'computed', '100 - queue_match (langsmith/monitoring_note.md, synthetic emails)')

# ---- trace_summary.csv: total_cost_eur is already USD x 0.9123, so it is NOT converted again ----
tr = pd.read_csv(REPO / 'langsmith/trace_summary.csv').set_index('set')
per_draft_eur = tr.loc['timing_10', 'total_cost_eur'] / tr.loc['timing_10', 'n']
A['ai_cost_per_draft'] = (round(per_draft_eur, 6), 'EUR/draft', 'measured (USD by LangSmith), EUR estimate', 'langsmith/trace_summary.csv timing_10 total_cost_eur / n')

# ---- Shopzilla cross-check: connected_handling_time (unit not documented, assumed seconds) ----
sz = None
med = None
if RAW_OK:
    import csv
    vals, email_n = [], 0
    with open(RAW, newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            t = r['connected_handling_time'].strip()
            if t != '':
                vals.append(float(t) / 60)
                if r['channel_name'].strip().lower() == 'email':
                    email_n += 1
    vals.sort()
    def pct(p):
        k = (len(vals) - 1) * p / 100
        lo = int(k)
        hi = min(lo + 1, len(vals) - 1)
        return vals[lo] + (vals[hi] - vals[lo]) * (k - lo)
    sz = {'n': len(vals), 'email_n': email_n, 'p5': pct(5), 'median': pct(50), 'p95': pct(95), 'max': vals[-1]}
    med = sz['median']
    src = 'data/raw/shopzilla_cs_satisfaction.csv connected_handling_time, n=' + str(sz['n']) + ' filled rows, unit assumed seconds, mostly calls; ' + KAGGLE
    A['shopzilla_handling_p5'] = (round(sz['p5'], 1), 'min (unit assumed seconds)', 'computed', src)
    A['shopzilla_handling_median'] = (round(med, 1), 'min (unit assumed seconds)', 'computed', src)
    A['shopzilla_handling_p95'] = (round(sz['p95'], 1), 'min (unit assumed seconds)', 'computed', src)
    print('Shopzilla handling time (min):', {k: round(x, 1) for k, x in sz.items()})

# ---- timing_log.csv (10 manual + 10 ai_assisted rows; the reported average in every row) ----
notes = {
 'manual': 'estimate (Ayhan), average, not a per-email measurement' + ('; Shopzilla handling time median ~' + str(round(med)) + ' min (n=' + str(sz['n']) + ')' if sz else ''),
 'ai_assisted': 'measured average (self-reported by Ayhan), not a per-email measurement; done second (learning bias)',
}
mins = {'manual': man, 'ai_assisted': ai}
rows = [[i, m, mins[m], notes[m]] for m in ['manual', 'ai_assisted'] for i in TIMING_IDS]
pd.DataFrame(rows, columns=['email_id', 'mode', 'minutes', 'notes']).to_csv(REPO / 'data/timing_log.csv', index=False)

# ---- 1 Build (prototype is sunk; payback uses pilot + rollout only) ----
day_rate = v['rate_h'] * v['hours_day']
phases = [('Prototype', v['days_prototype']), ('Pilot', v['days_pilot']), ('Rollout', v['days_rollout'])]
build_cost = {n: d * day_rate for n, d in phases}
build_total = sum(build_cost.values())
build_fwd = build_cost['Pilot'] + build_cost['Rollout']

# ---- 2 AI calls ----
ai_month = per_draft_eur * vol

# ---- 3 Tools per month (USD converted with usd_eur) ----
fx = v['usd_eur']
tools = [
 ('n8n Cloud Pro', 1, v['n8n_eur']),
 ('LangSmith Plus', v['ls_seats'], v['ls_usd'] * v['ls_seats'] * fx),
 ('Airtable Team', v['at_seats'], v['at_usd'] * v['at_seats'] * fx),
 ('Tableau Public', 0, v['tableau_eur']),
]
tools_month = sum(t[2] for t in tools)
executions = vol * v['n8n_exec_per_email']
exec_ok = executions < v['n8n_limit']

# ---- 4 Value ----
hourly = v['salary'] * v['overhead'] / v['hours_month']

def calc(manual):
    saved = manual - ai
    hours = vol * saved / 60
    value = hours * hourly
    net = value - ai_month - tools_month
    pay = build_fwd / net if net > 0 else None
    return saved, hours, value, net, pay

main = calc(man)
saved, hours, value, net, payback = main
print('day rate', day_rate, '| build total', build_total, '| build for payback', build_fwd)
print('ai/draft', per_draft_eur, '| ai/month', round(ai_month, 2), '| tools/month', round(tools_month, 2), '| hourly', round(hourly, 3))
print('main: hours', round(hours, 1), '| value', round(value, 2), '| net', round(net, 2), '| payback', payback and round(payback, 1))

def eur(x):
    return f'{x:,.2f}'

def pay_txt(pb):
    return 'n/a (no payback)' if pb is None else f'{pb:.1f}'

# ---- write cost_analysis.md ----
L = ['# Cost analysis (R1-06)', '',
 'Spreekorb is an invented company. Every number is computed, cited, or labelled estimate / synthetic. The timing emails are the synthetic set spreekorb_emails_60.csv. Volume and the handling-time cross-check come from Shopzilla (' + KAGGLE + '), a real-data proxy, not Spreekorb.', '',
 '## Assumptions', '',
 '| item | value | unit | label | source |', '|---|---|---|---|---|']
for k, (val, unit, label, s_) in A.items():
    L.append(f'| {k} | {val} | {unit} | {label} | {s_} |')

L += ['', '## 1. Build (day rate x days)', '',
 f'Day rate = {v["rate_h"]} EUR/h x {v["hours_day"]} h = {day_rate} EUR/day (estimate).', '',
 '| phase | days (estimate) | cost EUR (estimate) | in payback |', '|---|---|---|---|']
for n, d in phases:
    L.append(f'| {n} | {d} | {eur(d * day_rate)} | {"no, sunk (done in Round 1)" if n == "Prototype" else "yes"} |')
L += [f'| Total | {sum(d for n, d in phases)} | {eur(build_total)} | |',
 f'| Build for payback (pilot + rollout) | {v["days_pilot"] + v["days_rollout"]} | {eur(build_fwd)} | |', '',
 'Prototype was done in Round 1 (n8n POC, LangSmith, Tableau) and is sunk. Payback uses pilot + rollout build cost only. Days are own estimates without source.', '']

L += ['## 2. AI calls', '',
 f'AI cost per draft: {per_draft_eur:.6f} EUR (timing_10: total_cost_eur {tr.loc["timing_10", "total_cost_eur"]} / {tr.loc["timing_10", "n"]} drafts from langsmith/trace_summary.csv; USD measured by LangSmith, the EUR column is already converted at USDEUR {fx}, an estimate, and is not converted again).', '',
 f'Monthly = {per_draft_eur:.6f} x {vol:,} emails = {eur(ai_month)} EUR/month (computed).', '']

L += ['## 3. Tools per month (pilot)', '', '| tool | seats | EUR/month (estimate) |', '|---|---|---|']
for n, s, c in tools:
    L.append(f'| {n} | {s} | {eur(c)} |')
L += [f'| Total | | {eur(tools_month)} |', '',
 f'- USD prices converted at {fx} (tradingview.com USDEUR, 27 Nov 2026). Seat counts are estimates.',
 f'- n8n: {executions:,} executions/month (assumption: one email = one execution) vs {v["n8n_limit"]:,} included: {"OK, below the limit" if exec_ok else "ABOVE the limit"}. Price taken as 60 EUR monthly billing; about 50 EUR if billed annually. Check before buying. Self-hosted n8n is a rollout option (not costed).',
 '- Airtable is used: the review queue is an Airtable base (n8n/airtable_setup.md), so it is costed here.',
 '- Tableau Public is free but public data only. Real Spreekorb data cannot go on Tableau Public; a Tableau Cloud licence would be needed, price: source to add.', '']

L += ['## 4. Value', '',
 '### Timing evidence (10 held-out emails, data/timing_log.csv)', '',
 f'- Manual: {man} min per email on average. Label: estimate (Ayhan), revised from 20 to 15 min on 27 Nov 2026 after the Shopzilla cross-check; no measured basis.',
 f'- AI-assisted (draft + review + edit + send): {ai} min per email on average, self-reported average by Ayhan.', '',
 '### Timing test limits', '',
 '- One person did the test, and that person also built the system.',
 '- Only 10 synthetic held-out emails (spreekorb_emails_60.csv).',
 '- No blinding.',
 '- No counterbalancing: AI-assisted was done second on the same emails, so there is a learning bias (AI time probably understated, saving overstated).',
 '- Averages only; per-email minutes were not recorded.',
 '- Pilot fix: several agents, per-email timing, counterbalanced order.']
if sz:
    L += ['', '### Cross-check of the manual time (Shopzilla handling time)', '',
     f'Shopzilla ({KAGGLE}) column connected_handling_time is filled for only {sz["n"]:,} rows; the unit is not documented, assumed seconds. Computed in minutes: 5th percentile {sz["p5"]:.1f}, median {med:.1f}, 95th percentile {sz["p95"]:.1f}, max {sz["max"]:.1f}.',
     f'Rows are mostly Inbound / Outcall calls, all channels mixed. Email rows: n={sz["email_n"]}, anecdotal, no email rate is reported.',
     f'The median of {med:.1f} min (mostly calls) neither proves nor refutes the {man} min estimate (Ayhan). It is used as a conservative case.']

# main case vs conservative case, same weight, side by side
cases = [(f'main (manual {man} min, estimate Ayhan)', main)]
if sz:
    cases.append((f'conservative (manual {med:.1f} min, Shopzilla median, computed)', calc(med)))

def row(name, f):
    return '| ' + name + ' | ' + ' | '.join(f(c[1]) for c in cases) + ' |'

L += ['', f'### Formula (main case, manual = {man} min)', '',
 f'- Minutes saved per email = {man} - {ai} = {saved}',
 f'- Hours saved per month = {vol:,} x {saved} / 60 = {hours:,.1f} h',
 f'- Agent cost per hour = {v["salary"]:,} x {v["overhead"]} / {v["hours_month"]} = {hourly:.2f} EUR/h (estimate)',
 f'- Value per month = {hours:,.1f} x {hourly:.2f} = {eur(value)} EUR (estimate)',
 f'- Net per month = value - AI calls - tools = {eur(value)} - {eur(ai_month)} - {eur(tools_month)} = {eur(net)} EUR',
 f'- Payback = build for payback (pilot + rollout) / net per month = {eur(build_fwd)} / {eur(net)} = {pay_txt(payback)} months (estimate)',
 '', 'Time saved is capacity (agents answer more emails, faster), not an automatic headcount cut.', '',
 '### Main case vs conservative case', '',
 '| metric | ' + ' | '.join(c[0] for c in cases) + ' |', '|---|' + '---|' * len(cases),
 row('minutes saved per email', lambda r: f'{r[0]:.1f}'),
 row('hours saved per month', lambda r: f'{r[1]:,.1f}'),
 row('value EUR/month', lambda r: eur(r[2])),
 row('net EUR/month (after AI calls and tools)', lambda r: eur(r[3])),
 row('payback months (pilot + rollout build)', lambda r: pay_txt(r[4])),
 '', '### Sensitivity (AI-assisted fixed at the self-reported average)', '',
 'Levels 3, 10, 20, 26 min are illustrative, not measured.', '',
 '| manual min | saved min | hours saved/month | value EUR/month | net EUR/month | payback months |', '|---|---|---|---|---|---|']
levels = [(x, f'{x} (illustrative)') for x in [3, 10, 20, 26] if x != man] + [(man, f'{man} (Ayhan)')]
if sz:
    levels.append((med, f'{med:.1f} (Shopzilla median, computed)'))
for m, lab in sorted(levels):
    s, h, val, nt, pb = calc(m)
    L.append(f'| {lab} | {s:.1f} | {h:,.1f} | {eur(val)} | {eur(nt)} | {pay_txt(pb)} |')
L += ['', f'At {ai} min AI-assisted and 3 min manual the saving is negative: AI would cost more time than it saves. The business case depends on the manual time, which is still an estimate.', '',
 '### Second value: faster first response through triage (qualitative)', '',
 'Auto triage routes each email to the right queue / department, so customers should get a first response sooner (SLA). No SLA improvement number is claimed.',
 f'- Held-out queue_match is {v["queue_match"]}% on 20 synthetic emails (langsmith/monitoring_note.md), so about {re_route}% still need manual re-routing.',
 '- A human approves every draft; nothing is sent automatically.',
 '- Link to the proposed SLA target: first response within 24 h for 90% of emails (research/sector_research.md).', '']
(REPO / 'cost_estimation/cost_analysis.md').write_text(NL.join(L) + NL, encoding='utf-8')

# ---- write timeline_estimate.md ----
T = ['# Timeline estimate (R1-06)', '',
 'Durations and costs are estimates (see the assumptions table in cost_analysis.md). No calendar dates are committed; weeks are counted from the stated start. Cost = days x day rate.', '']
plan = [
 ('Prototype', 'Done in Round 1', f'about {v["weeks_prototype"]} weeks (done)', 'n8n proof of concept, LangSmith monitoring, Tableau dashboard, synthetic data only.',
  'Triage + draft run end to end; held-out check and timing sample done; human approves every draft.', 'AI consultant'),
 ('Pilot', 'Starts after the Round 1 decision (week 1)', f'about {v["weeks_pilot"]} weeks', 'Real-email test on 1 team: n8n + mailbox + review UI, DPA and works-council preparation.',
  'DPA + EU region + works agreement signed; timing re-measured per email with several agents and counterbalanced order; CSAT and SLA tracked at team level (minimum group 5).', 'Project lead with data protection officer and works council'),
 ('Rollout', 'Starts after the pilot exit criteria are met', f'about {v["weeks_rollout"]} weeks', 'All email teams; hosting decision (n8n Cloud or self-hosted); Tableau Cloud licence if real data is shown.',
  'Pilot results reviewed by management; team-level SLA and CSAT reported; support handover done.', 'Head of customer service with IT owner'),
]
for (n, start, dur, scope, exitc, owner), (pn, pd_days) in zip(plan, phases):
    sunk = ' (sunk, done in Round 1)' if pn == 'Prototype' else ''
    T += [f'## {n}', '', f'- Start: {start}', f'- Duration (estimate): {dur}; effort {pd_days} days',
          f'- Scope: {scope}', f'- Exit criteria: {exitc}', f'- Owner (role): {owner}',
          f'- Cost (estimate): {pd_days} days x {day_rate} EUR = {eur(build_cost[pn])} EUR{sunk}', '']
(REPO / 'cost_estimation/timeline_estimate.md').write_text(NL.join(T) + NL, encoding='utf-8')
print('Written:', OUTPUTS)

Shopzilla handling time (min): {'n': 242, 'email_n': 4, 'p5': 2.8, 'median': 7.1, 'p95': 15.3, 'max': 33.1}
day rate 760 | build total 26600 | build for payback 19000
ai/draft 0.0003469 | ai/month 1.05 | tools/month 185.9 | hourly 19.875
main: hours 502.8 | value 9993.81 | net 9806.87 | payback 1.9
Written: ['data/timing_log.csv', 'cost_estimation/cost_analysis.md', 'cost_estimation/timeline_estimate.md']


In [4]:
# 4 Verify: each acceptance item prints PASS/FAIL
import re

log = pd.read_csv(REPO / 'data/timing_log.csv', dtype=str).fillna('')
ok('timing_log keeps 4 columns', list(log.columns) == ['email_id', 'mode', 'minutes', 'notes'])
ok('10 manual rows', (log['mode'] == 'manual').sum() == 10)
ok('10 ai_assisted rows', (log['mode'] == 'ai_assisted').sum() == 10)
for m in ['manual', 'ai_assisted']:
    ok('ids = 10 timing_flag held-out emails (' + m + ')', sorted(log[log['mode'] == m]['email_id']) == TIMING_IDS)
ok('minutes numeric', pd.to_numeric(log['minutes'], errors='coerce').notna().all())
ok('notes non-empty', (log['notes'].astype(str).str.strip() != '').all())
ok('every note says average and not a per-email', log['notes'].str.contains('average').all() and log['notes'].str.contains('not a per-email').all())

cost = (REPO / 'cost_estimation/cost_analysis.md').read_text(encoding='utf-8')
for h in ['## 1. Build', '## 2. AI calls', '## 3. Tools', '## 4. Value', '### Timing test limits']:
    ok('cost section: ' + h, h in cost)
ok('day rate present', 'day rate' in cost.lower())

# assumptions table: every row has a label and a source
sec = cost.split('## Assumptions')[1].split(NL + '## ')[0]
table = [r for r in sec.split(NL) if r.startswith('|')][2:]
cells = [[c.strip() for c in r.strip('|').split('|')] for r in table]
LABELS = ('measured', 'estimate', 'synthetic', 'computed')
ok('assumptions table has rows', len(cells) > 10)
ok('every row has label + source', all(len(c) == 5 and c[3].split(' ')[0] in LABELS and c[4] != '' for c in cells))
if RAW_OK:
    ok('Shopzilla cross-check row, label computed', any(c[0] == 'shopzilla_handling_median' and c[3].startswith('computed') for c in cells))
else:
    print('WARN - Shopzilla cross-check row skipped (raw file missing)')

# AI cost: per draft vs the CSV, and per month = per draft x volume
tr = pd.read_csv(REPO / 'langsmith/trace_summary.csv').set_index('set')
expected = tr.loc['timing_10', 'total_cost_eur'] / tr.loc['timing_10', 'n']
found = float(re.search(r'AI cost per draft: ([0-9.]+) EUR', cost).group(1))
ok('AI cost per draft matches trace_summary', abs(found - expected) < 1e-6)
monthly = float(re.search(r'Monthly = .* = ([0-9,.]+) EUR/month', cost).group(1).replace(',', ''))
ok('AI cost per month = per draft x volume', abs(monthly - found * vol) < 0.01)

# key numbers appear in the md
ok('build total in md', eur(build_total) in cost)
ok('tools total in md', eur(tools_month) in cost)
ok('value in md', eur(value) in cost)
ok('queue_match 85 in md', '85' in cost)
ok('volume 3,017 in md', '3,017' in cost or '3017' in cost)
if RAW_OK:
    ok('Shopzilla n in md', str(sz['n']) in cost)

tl = (REPO / 'cost_estimation/timeline_estimate.md').read_text(encoding='utf-8')
pos = [tl.find('## Prototype'), tl.find('## Pilot'), tl.find('## Rollout')]
ok('timeline order prototype > pilot > rollout', -1 not in pos and pos == sorted(pos))

# key strings: a long sk- token, or the env-file name (built so this cell does not contain it)
env_name = '.' + 'env'
for name, text in [('cost_analysis', cost), ('timeline', tl), ('timing_log', log.to_csv())]:
    found_key = re.search(r'sk-[A-Za-z0-9]{20,}', text) or re.search(re.escape(env_name) + r'\b', text)
    ok('no key strings in ' + name, not found_key)

PASS - timing_log keeps 4 columns
PASS - 10 manual rows
PASS - 10 ai_assisted rows
PASS - ids = 10 timing_flag held-out emails (manual)
PASS - ids = 10 timing_flag held-out emails (ai_assisted)
PASS - minutes numeric
PASS - notes non-empty
PASS - every note says average and not a per-email
PASS - cost section: ## 1. Build
PASS - cost section: ## 2. AI calls
PASS - cost section: ## 3. Tools
PASS - cost section: ## 4. Value
PASS - cost section: ### Timing test limits
PASS - day rate present
PASS - assumptions table has rows
PASS - every row has label + source
PASS - Shopzilla cross-check row, label computed
PASS - AI cost per draft matches trace_summary
PASS - AI cost per month = per draft x volume
PASS - build total in md
PASS - tools total in md
PASS - value in md
PASS - queue_match 85 in md
PASS - volume 3,017 in md
PASS - Shopzilla n in md
PASS - timeline order prototype > pilot > rollout
PASS - no key strings in cost_analysis
PASS - no key strings in timeline
PASS - no key strings i

In [5]:
# 5 Unchanged: files we do not own must have the same hash as before
for p in PROTECTED:
    ok('unchanged: ' + p, sha(p) == HASH_BEFORE[p])

PASS - unchanged: langsmith/trace_summary.csv
PASS - unchanged: data/processed/spreekorb_emails_60.csv
PASS - unchanged: langsmith/monitoring_note.md
PASS - unchanged: langsmith/human_review.csv
PASS - unchanged: data/raw/shopzilla_cs_satisfaction.csv


In [ ]:
# 6 Undo: restores the backup, only if RUN_UNDO = True
RUN_UNDO = False
if RUN_UNDO:
    for p in OUTPUTS:
        if EXISTED[p]:
            shutil.copy2(BK / p, REPO / p)   # put the old file back
        else:
            (REPO / p).unlink(missing_ok=True)   # file did not exist before
    print('Undo done from', BK)
else:
    print('Undo skipped (RUN_UNDO = False)')

In [6]:
# 7 Git: print the commands, never run them (Ayhan commits)
print('git add data/timing_log.csv cost_estimation/cost_analysis.md cost_estimation/timeline_estimate.md notebooks/R1-06_cost_timeline.ipynb')
print('git commit -m "R1-06 timing, cost analysis, timeline"')

git add data/timing_log.csv cost_estimation/cost_analysis.md cost_estimation/timeline_estimate.md notebooks/R1-06_cost_timeline.ipynb
git commit -m "R1-06 timing, cost analysis, timeline"
